# Has the structure of risk in financial markets changed since COVID-19?

## Part 0 — Packages and helper functions

We import only what is needed. The `arch` package is used for validation of our hand-written GARCH estimator. All core models are estimated manually following the approach in the course notebooks.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.stattools import adfuller
from arch import arch_model
from statsmodels.stats.diagnostic import breaks_cusumolsresid
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.graphics.tsaplots import plot_acf

ANNUALIZATION = 252

In [ ]:
DATA_PATH = "Data.xlsx"

prices = pd.read_excel(DATA_PATH, index_col=0, parse_dates=True)
prices.index.name = "Date"
prices = prices.sort_index().ffill()
prices = prices.where(prices > 0).ffill()

prices.tail()

## Part 1 — Data loading and sample construction

We select six assets from the dataset, one per distinct risk dimension, following the data selection logic of Chorro, Guégan & Ielpo (2012). Log-returns are computed as $r_t = \log(P_t / P_{t-1})$ and the sample is split at 24 February 2020.

In [ ]:
# One series per non-redundant risk dimension
# Excluded: yields (log-returns of near-zero yields are explosive),
# MSCI EM and Hang Seng (post-COVID correlation 0.82 → redundant),
# FX series (reflect rate and equity dynamics already captured),
# SMI (correlation with Eurostoxx 50 = 0.81)

ASSETS = ["S&P500", "Eurostoxx 50", "US IG Bonds", "US HY Bonds", "Gold", "Oil futures"]

prices = prices[ASSETS]
prices.tail()

### Missing values

Missing prices arise from different exchange calendars. We document the coverage per asset before forward-filling.

In [ ]:
print(f"{'Asset':<20} {'Missing':>8}  {'First valid':>12}  {'Last valid':>12}")
print("-" * 58)
for col in ASSETS:
    n  = prices[col].isna().sum()
    fv = prices[col].first_valid_index().date()
    lv = prices[col].last_valid_index().date()
    print(f"{col:<20} {n:>8,}  {str(fv):>12}  {str(lv):>12}")

### Log-returns and sample split

The COVID breakpoint is set at **24 February 2020** — the start of the first major equity sell-off week (S&P 500 −11% over the following 10 trading days). The crash period is included in the post-COVID sample: excluding it would bias structural-change tests toward zero.

In [ ]:
print(f"\nNon-NaN observations per asset")
print(f"{'Asset':<20} {'Full':>7}  {'Pre':>7}  {'Post':>7}")
print("-" * 46)
for col in ASSETS:
    print(f"{col:<20} {returns[col].notna().sum():>7,}  {pre[col].notna().sum():>7,}  {post[col].notna().sum():>7,}")

### Breakpoint selection

We propose 24 February 2020 as the structural break date on economic grounds — it marks the start of the first major COVID-driven equity sell-off. However, asserting a breakpoint without testing it is not rigorous. We apply a **Quandt-Andrews supremum-Wald test** over a candidate window (January 2020 – June 2020), which searches for the date that maximises the Wald statistic for a mean-shift in S&P 500 returns. The data-driven date either confirms our choice or replaces it.

In [ ]:


# Quandt-Andrews sup-Wald test on S&P 500 returns
# We test for a single structural break in the mean over the full sample
r_spx = returns["S&P500"].dropna()

X = add_constant(np.arange(len(r_spx)))
model = OLS(r_spx.values, X).fit()

# Sup-Wald: test each candidate date in the window, keep the date
# that maximises the F-statistic for a mean break
window_start = pd.Timestamp("2020-01-01")
window_end   = pd.Timestamp("2020-06-30")
candidate_idx = r_spx.index[(r_spx.index >= window_start) & (r_spx.index <= window_end)]

wald_stats = {}
for date in candidate_idx:
    d = (r_spx.index <= date).astype(int)
    X_break = np.column_stack([np.ones(len(r_spx)), d])
    res = OLS(r_spx.values, X_break).fit()
    # F-statistic for the dummy coefficient
    wald_stats[date] = res.tvalues[1]**2

break_date = max(wald_stats, key=wald_stats.get)
print(f"Proposed breakpoint : 2020-02-24")
print(f"Sup-Wald break date : {break_date.date()}")
print(f"Max Wald statistic  : {wald_stats[break_date]:.2f}")

In [ ]:
# Use the data-driven break date if it differs materially from our proposal
COVID_DATE = break_date
print(f"COVID_DATE set to: {COVID_DATE.date()}")

pre  = returns.loc[:COVID_DATE]
post = returns.loc[COVID_DATE + pd.Timedelta(days=1):]

print(f"\nPre-COVID   : {pre.index[0].date()} → {pre.index[-1].date()}  ({len(pre):,} obs)")
print(f"Post-COVID  : {post.index[0].date()} → {post.index[-1].date()}  ({len(post):,} obs)")

The sup-Wald test identifies **23 March 2020** as the structural break date — the day the Federal Reserve announced unlimited quantitative easing and the S&P 500 reached its COVID trough. This is the moment of maximum structural shock to risk markets, and we use it as our breakpoint. The pre-COVID sample therefore captures the full crash period up to and including the trough, and the post-COVID sample begins the day after.

In [ ]:
print(f"Non-NaN observations per asset")
print(f"{'Asset':<20} {'Full':>7}  {'Pre':>7}  {'Post':>7}")
print("-" * 46)
for col in ASSETS:
    print(f"{col:<20} {returns[col].notna().sum():>7,}  "
          f"{pre[col].notna().sum():>7,}  {post[col].notna().sum():>7,}")

## Part 2 — Stylized facts and pre/post comparison

Before estimating any model, we establish three things: (1) the data exhibits the three stylized facts that justify GARCH, (2) the headline pre/post descriptive statistics already reveal structural changes, and (3) an ARCH-LM test formally confirms conditional heteroskedasticity in all six series.

### Return series

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(ASSETS):
    ax = axes[i]
    ax.plot(returns.index, returns[col], lw=0.7, color="steelblue")
    ax.axvline(COVID_DATE, color="crimson", lw=1.2, linestyle="--", label="Break")
    ax.set_title(col)
    ax.set_ylabel("Log-return")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))

axes[0].legend(framealpha=0.5)
fig.suptitle("Daily log-returns — full sample", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Descriptive statistics

We report annualised mean, volatility, skewness, excess kurtosis, Jarque-Bera p-value, min and max for each asset. All series are expected to reject normality (Cont 2001).

In [ ]:
def summary_stats(r):
    if isinstance(r, pd.Series):
        r = r.to_frame()
    rows = []
    for col in r.columns:
        s = r[col].dropna()
        jb_stat, jb_p = stats.jarque_bera(s)
        adf_p = adfuller(s, autolag="AIC")[1]
        rows.append({
            "mean_ann"    : s.mean() * ANNUALIZATION,
            "vol_ann"     : s.std()  * np.sqrt(ANNUALIZATION),
            "skewness"    : round(float(stats.skew(s)), 3),
            "ex_kurtosis" : round(float(stats.kurtosis(s)), 3),
            "jb_pval"     : round(jb_p, 4),
            "min"         : round(s.min(), 4),
            "max"         : round(s.max(), 4),
            "adf_pval"    : round(adf_p, 4),
        })
    return pd.DataFrame(rows, index=r.columns)

summary_stats(returns).round(4)

### Pre vs post comparison

In [ ]:
stats_pre  = summary_stats(pre).add_suffix("_pre")
stats_post = summary_stats(post).add_suffix("_post")

comparison = pd.concat([stats_pre[["vol_ann_pre", "skewness_pre", "ex_kurtosis_pre", "jb_pval_pre"]],
                         stats_post[["vol_ann_post", "skewness_post", "ex_kurtosis_post", "jb_pval_post"]]], axis=1)
comparison.round(4)

### Volatility comparison — pre vs post

In [ ]:
vol_pre  = pre.std()  * np.sqrt(ANNUALIZATION)
vol_post = post.std() * np.sqrt(ANNUALIZATION)

x = np.arange(len(ASSETS))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - width/2, vol_pre,  width, label="Pre-COVID",  color="steelblue")
ax.bar(x + width/2, vol_post, width, label="Post-COVID", color="crimson", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(ASSETS, rotation=15)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.set_title("Annualised volatility — pre vs post COVID")
ax.legend()
plt.tight_layout()
plt.show()

### Static correlations — pre vs post

The correlation matrix is the first, unconditional look at co-movement. We compare pre and post side by side and compute the change matrix to identify the largest structural shifts.

In [ ]:
corr_pre  = pre.corr()
corr_post = post.corr()
corr_diff = corr_post - corr_pre

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
kw = dict(annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True, cbar=False)

sns_kw = {**kw}
import seaborn as sns
sns.heatmap(corr_pre,  ax=axes[0], **kw)
sns.heatmap(corr_post, ax=axes[1], **kw)
sns.heatmap(corr_diff, ax=axes[2], annot=True, fmt=".2f",
            cmap="RdBu_r", vmin=-0.5, vmax=0.5, square=True, cbar=False)

axes[0].set_title("Pre-COVID correlations")
axes[1].set_title("Post-COVID correlations")
axes[2].set_title("Change (post − pre)")
plt.tight_layout()
plt.show()

### Stylized fact 1 — volatility clustering

The ACF of squared returns should be large and persistent. The ACF of raw returns should be near zero. Both together confirm that volatility clusters in time, which motivates the GARCH framework (Bollerslev 1986).

In [ ]:
fig, axes = plt.subplots(len(ASSETS), 2, figsize=(14, 3 * len(ASSETS)))

for i, col in enumerate(ASSETS):
    r = returns[col].dropna()
    plot_acf(r,    ax=axes[i, 0], lags=40, alpha=0.05, title=f"{col} — ACF(r)")
    plot_acf(r**2, ax=axes[i, 1], lags=40, alpha=0.05, title=f"{col} — ACF(r²)")

plt.tight_layout()
plt.show()

### Stylized fact 2 — fat tails and non-normality

Jarque-Bera tests confirm all six series reject normality. This motivates the Student-t innovation distribution in Part 3, following Bollerslev (1987) and the distributional argument of Chorro, Guégan & Ielpo (2012).

In [ ]:
print(f"{'Asset':<20}  {'JB statistic':>14}  {'p-value':>9}  {'Reject H0':>10}")
print("-" * 58)
for col in ASSETS:
    r = returns[col].dropna()
    jb, p = stats.jarque_bera(r)
    reject = "Yes ***" if p < 0.01 else ("Yes *" if p < 0.05 else "No")
    print(f"{col:<20}  {jb:>14,.1f}  {p:>9.4f}  {reject:>10}")

### Stylized fact 3 — formal ARCH-LM test

Before fitting any GARCH model, we must verify that conditional heteroskedasticity is present in the data. The Engle (1982) ARCH-LM test tests H₀: no ARCH effects. All six series must reject at the 1% level to formally justify the GARCH framework.

In [ ]:
print(f"{'Asset':<20}  {'LM statistic':>13}  {'p-value':>9}  {'Reject H0':>10}")
print("-" * 58)
for col in ASSETS:
    r = returns[col].dropna()
    lm, p, _, _ = het_arch(r, nlags=10)
    reject = "Yes ***" if p < 0.01 else ("Yes *" if p < 0.05 else "No")
    print(f"{col:<20}  {lm:>13.2f}  {p:>9.4f}  {reject:>10}")

### Preliminary PCA on raw returns

Before GARCH filtering, we take a first look at the factor structure. PCA on raw returns mixes volatility differences with correlation structure — this will be refined in Part 6 using GARCH-filtered residuals. Here we simply document how much variance the first three principal components explain, pre and post COVID.

An increase in the variance explained by PC1 post-COVID would indicate that the six assets started moving more in tandem — risk became more concentrated in a single common factor. We will revisit this with GARCH-filtered residuals in Part 6 to confirm whether the shift is a genuine correlation phenomenon or a volatility artifact.

## Part 3 — Univariate GARCH (Layer 1: Risk Levels)

We estimate a volatility model for each of the six assets, on the full sample and on each subperiod separately. The goal is to characterise how individual risk levels changed post-COVID: did volatility become more persistent? Did the leverage effect strengthen or weaken? Did the tail of the conditional distribution get fatter?

### 3.1 Model specification

We use a **GJR-GARCH(1,1)** with **Student-t innovations**:

$$r_t = \mu + \varepsilon_t, \qquad \varepsilon_t = \sigma_t z_t, \qquad z_t \sim t_\nu$$

$$\sigma^2_t = \omega + \alpha \varepsilon^2_{t-1} + \gamma \mathbf{1}[\varepsilon_{t-1} < 0]\, \varepsilon^2_{t-1} + \beta \sigma^2_{t-1}$$

**Why GJR-GARCH?** The standard GARCH(1,1) is symmetric — it treats positive and negative shocks identically. The GJR extension adds $\gamma$, which captures the leverage effect (Stylized Fact 3): negative shocks increase volatility more than positive shocks of the same magnitude (Glosten, Jagannathan & Runkle 1993).

**Why Student-t?** The Jarque-Bera tests in Part 2 reject normality for all six series. Under the QMLE framework, using the correct non-normal likelihood is more efficient than the Gaussian approximation — the efficiency gain increases with the degree of departure from normality (Engle & González-Rivera 1991). Given the extreme kurtosis in our data, we maximise the Student-t log-likelihood directly.

**Why not GH?** The Generalised Hyperbolic distribution (Chorro, Guégan & Ielpo 2012) would be the natural extension, but its five parameters require at least 4,000 observations for reliable estimation. Our post-COVID subsample has ~1,600 observations. The Student-t provides a parsimonious and interpretable approximation: $\nu$ directly measures tail thickness and is comparable across subperiods.

**Key implication from Ielpo (2014):** when the innovation distribution is sufficiently flexible, $\gamma$ often becomes statistically insignificant — the distribution absorbs the asymmetry that $\gamma$ would otherwise capture. Testing whether $\gamma$ is significant pre- and post-COVID is therefore itself a research finding.

### 3.2 Model selection — per asset specification

Before estimating any GARCH model, we let the data determine the correct specification for each asset. Three questions must be answered:

1. **Does the mean equation need an AR(1) term?** — Ljung-Box on raw returns
2. **Is the leverage effect present?** — we will test γ significance post-estimation, but we include GJR for all assets and let the data decide
3. **What GARCH order is needed?** — Ljung-Box on squared returns after a preliminary ARCH(1) filter; if autocorrelation remains at long lags, higher order may be needed

This mirrors the lag-selection logic used in VAR estimation.

In [ ]:
# Test 1: AR(1) in the mean — Ljung-Box on raw returns
print("=" * 62)
print("TEST 1 — Autocorrelation in raw returns (mean equation)")
print("H0: no autocorrelation  |  reject → AR(1) needed")
print("=" * 62)
print(f"{'Asset':<20}  {'LB(5)':>8}  {'LB(10)':>8}  {'LB(20)':>8}  {'AR(1)?':>8}")
print("-" * 62)

ar1_needed = {}
for col in ASSETS:
    r  = returns[col].dropna()
    lb = acorr_ljungbox(r, lags=[5, 10, 20], return_df=True)
    p5, p10, p20 = lb["lb_pvalue"].values
    need = p5 < 0.05 or p10 < 0.05
    ar1_needed[col] = need
    flag = "Yes ***" if need else "No"
    print(f"{col:<20}  {p5:>8.4f}  {p10:>8.4f}  {p20:>8.4f}  {flag:>8}")

In [ ]:
# Test 2: ARCH effects in raw returns — confirms GARCH is needed
print("=" * 62)
print("TEST 2 — ARCH effects in raw returns")
print("H0: no ARCH effects  |  reject → GARCH needed")
print("=" * 62)
print(f"{'Asset':<20}  {'LM stat':>9}  {'p-value':>9}  {'GARCH?':>8}")
print("-" * 62)

for col in ASSETS:
    r = returns[col].dropna()
    lm, p, _, _ = het_arch(r, nlags=10)
    flag = "Yes ***" if p < 0.01 else ("Yes *" if p < 0.05 else "No")
    print(f"{col:<20}  {lm:>9.2f}  {p:>9.4f}  {flag:>8}")

In [ ]:
# Test 3: GARCH order — Ljung-Box on squared returns at multiple lags
# High autocorrelation at long lags suggests richer dynamics may be needed
print("=" * 62)
print("TEST 3 — Autocorrelation in squared returns (variance equation)")
print("H0: no autocorrelation  |  persistent rejection → higher order")
print("=" * 62)
print(f"{'Asset':<20}  {'LB(5)':>8}  {'LB(10)':>8}  {'LB(20)':>8}  {'LB(40)':>8}")
print("-" * 62)

for col in ASSETS:
    r  = returns[col].dropna()
    r2 = r**2
    lb = acorr_ljungbox(r2, lags=[5, 10, 20, 40], return_df=True)
    p5, p10, p20, p40 = lb["lb_pvalue"].values
    print(f"{col:<20}  {p5:>8.4f}  {p10:>8.4f}  {p20:>8.4f}  {p40:>8.4f}")

In [ ]:
# All assets need AR(1) based on Test 1
ar1_needed = {col: True for col in ASSETS}

print("=" * 62)
print("SPECIFICATION SUMMARY — locked")
print("=" * 62)
print(f"{'Asset':<20}  {'Mean eq.':>12}  {'Var. eq.':>12}  {'Innov.':>10}")
print("-" * 62)
for col in ASSETS:
    print(f"{col:<20}  {'AR(1)-GJR':>12}  {'GARCH(1,1)':>12}  {'Student-t':>10}")
print("-" * 62)
print("Justification:")
print("  AR(1)      — LB test rejects H0 for all 6 series (Test 1)")
print("  GARCH(1,1) — ARCH-LM rejects H0 for all 6 series (Test 2)")
print("               LB on r² persistent to lag 40 → (1,1) sufficient (Test 3)")
print("  GJR        — captures leverage effect (Stylized Fact 3)")
print("  Student-t  — JB rejects normality for all 6 series (Part 2)")

The specification is now locked per asset based on the data. We proceed to estimation in section 3.3.

### GARCH estimation functions

In [ ]:
def nu_from_lognu(log_nu):
    return 2.0 + np.exp(log_nu)

def lognu_from_nu(nu):
    return np.log(nu - 2.0)


def gjr_garch_filter(params, returns, ar1=False, ar2=False):
    """
    Parse params and run the GJR-GARCH(1,1) recursion.
    Returns sigma2, standardised residuals z, and nu.
    """
    idx   = 0
    mu    = params[idx]; idx += 1
    phi1  = params[idx] if (ar1 or ar2) else 0.0
    if ar1 or ar2: idx += 1
    phi2  = params[idx] if ar2 else 0.0
    if ar2: idx += 1
    omega = params[idx]; idx += 1
    alpha = params[idx]; idx += 1
    gamma = params[idx]; idx += 1
    beta  = params[idx]; idx += 1
    nu    = nu_from_lognu(params[idx])

    T         = len(returns)
    sigma2    = np.empty(T)
    eps       = np.empty(T)
    eps[0]    = 0.0
    sigma2[0] = np.var(returns)

    for t in range(1, T):
        mean_t    = mu + phi1 * returns[t-1]
        if ar2 and t >= 2:
            mean_t += phi2 * returns[t-2]
        eps[t]    = returns[t] - mean_t
        indicator = 1.0 if eps[t-1] < 0 else 0.0
        sigma2[t] = (omega
                     + alpha * eps[t-1]**2
                     + gamma * indicator * eps[t-1]**2
                     + beta  * sigma2[t-1])
        sigma2[t] = max(sigma2[t], 1e-8)

    z = eps / np.sqrt(sigma2)
    return sigma2, z, nu


def gjr_garch_loglik(params, returns, ar1=False, ar2=False):
    """
    Negative Student-t log-likelihood for GJR-GARCH(1,1).
    Uses Bollerslev (1987) formulation with variance-correct standardisation.
    """
    # parse nu for constraint check
    nu = nu_from_lognu(params[-1])

    # soft constraints
    idx   = 0 + (1 if ar1 or ar2 else 0) + (1 if ar2 else 0)
    omega = params[1 + (1 if ar1 or ar2 else 0) + (1 if ar2 else 0)]
    alpha = params[2 + (1 if ar1 or ar2 else 0) + (1 if ar2 else 0)]
    gamma = params[3 + (1 if ar1 or ar2 else 0) + (1 if ar2 else 0)]
    beta  = params[4 + (1 if ar1 or ar2 else 0) + (1 if ar2 else 0)]

    if omega <= 0 or alpha < 0 or gamma < 0 or beta < 0:
        return 1e10
    if alpha + beta + 0.5*gamma >= 1:
        return 1e10
    if nu <= 2:
        return 1e10

    sigma2, z, nu = gjr_garch_filter(params, returns, ar1=ar1, ar2=ar2)

    if np.any(sigma2 <= 0) or not np.all(np.isfinite(sigma2)):
        return 1e10

    # Bollerslev (1987) Student-t log-likelihood
    # z_t ~ t(nu) with variance 1, so scale = sqrt((nu-2)/nu)
    from scipy.special import gammaln
    c  = gammaln((nu+1)/2) - gammaln(nu/2) - 0.5*np.log(np.pi*(nu-2))
    ll = c - 0.5*np.log(sigma2) - ((nu+1)/2)*np.log(1 + z**2/(nu-2))

    if not np.all(np.isfinite(ll)):
        return 1e10

    return -np.sum(ll)


def fit_gjr_garch(returns, label="", ar1=False, ar2=False):
    r    = np.asarray(returns.dropna(), dtype=float)
    var0 = np.var(r)
    lognu_starts = [lognu_from_nu(nu) for nu in [4.0, 6.0, 8.0, 12.0, 20.0]]

    if ar2:
        base = [
            [r.mean(),  0.05,  0.02, var0*0.05, 0.05, 0.05, 0.90],
            [r.mean(),  0.10,  0.05, var0*0.05, 0.08, 0.08, 0.85],
            [r.mean(), -0.05,  0.05, var0*0.10, 0.10, 0.10, 0.80],
            [r.mean(),  0.02, -0.02, var0*0.02, 0.03, 0.05, 0.93],
        ]
        bounds = [(None,None), (-0.3,0.3), (-0.3,0.3),
                  (1e-9,None), (1e-6,0.5), (1e-6,0.5),
                  (1e-6,0.9999), (None,None)]
    elif ar1:
        base = [
            [r.mean(),  0.0,  var0*0.05, 0.05, 0.05, 0.90],
            [r.mean(),  0.05, var0*0.05, 0.08, 0.08, 0.85],
            [r.mean(), -0.05, var0*0.10, 0.10, 0.10, 0.80],
            [r.mean(),  0.02, var0*0.02, 0.03, 0.05, 0.93],
        ]
        bounds = [(None,None), (-0.3,0.3), (1e-9,None),
                  (1e-6,0.5), (1e-6,0.5), (1e-6,0.9999), (None,None)]
    else:
        base = [
            [r.mean(), var0*0.05, 0.05, 0.05, 0.90],
            [r.mean(), var0*0.05, 0.08, 0.08, 0.85],
            [r.mean(), var0*0.10, 0.10, 0.10, 0.80],
            [r.mean(), var0*0.02, 0.03, 0.05, 0.93],
        ]
        bounds = [(None,None), (1e-9,None),
                  (1e-6,0.5), (1e-6,0.5), (1e-6,0.9999), (None,None)]

    starting_points = [b + [lnu] for b in base for lnu in lognu_starts]

    best_result = None
    best_ll     = np.inf

    for p0 in starting_points:
        try:
            res = minimize(gjr_garch_loglik, p0, args=(r, ar1, ar2),
                           method="L-BFGS-B", bounds=bounds,
                           options={"maxiter": 10000, "ftol": 1e-14, "gtol": 1e-8})
            if res.fun < best_ll and np.isfinite(res.fun):
                best_ll     = res.fun
                best_result = res
        except Exception:
            continue

    if best_result is None:
        raise RuntimeError(f"All optimisations failed for {label}")
    if not best_result.success:
        print(f"  [WARNING] {label} — optimizer did not fully converge")

    sigma2, z, nu = gjr_garch_filter(best_result.x, r, ar1=ar1, ar2=ar2)

    # unpack cleanly via filter (avoids indexing bugs)
    p = best_result.x
    i = 0
    mu   = p[i]; i += 1
    phi1 = p[i] if (ar1 or ar2) else 0.0
    if ar1 or ar2: i += 1
    phi2 = p[i] if ar2 else 0.0
    if ar2: i += 1
    omega = p[i]; i += 1
    alpha = p[i]; i += 1
    gamma = p[i]; i += 1
    beta  = p[i]; i += 1

    persistence = alpha + beta + 0.5 * gamma
    uncond_vol  = np.sqrt(omega / max(1 - persistence, 1e-8)) * np.sqrt(ANNUALIZATION)
    half_life   = np.log(0.5) / np.log(persistence) if persistence < 1 else np.inf

    return {
        "params"     : {"mu": mu, "phi1": phi1, "phi2": phi2,
                        "omega": omega, "alpha": alpha,
                        "gamma": gamma, "beta": beta, "nu": nu},
        "sigma2"     : sigma2,
        "z"          : z,
        "returns"    : r,
        "loglik"     : -best_result.fun,
        "persistence": round(persistence, 5),
        "uncond_vol" : round(uncond_vol, 4),
        "half_life"  : round(half_life, 1),
        "label"      : label,
        "converged"  : best_result.success,
        "ar1"        : ar1,
        "ar2"        : ar2,
    }

In [ ]:
# S&P500 and US IG still show LB(z) failure with AR(1)
# Test whether AR(2) removes the remaining autocorrelation in the mean

for col in ["S&P500", "US IG Bonds"]:
    r  = returns[col].dropna()
    # fit AR(2) residuals manually
    phi1 = np.corrcoef(r[1:], r[:-1])[0,1]
    phi2 = np.corrcoef(r[2:], r[:-2])[0,1]
    resid = r.values[2:] - phi1*r.values[1:-1] - phi2*r.values[:-2]
    lb = acorr_ljungbox(resid, lags=[5, 10, 20], return_df=True)
    print(f"\n{col} — LB on AR(2) residuals:")
    print(lb["lb_pvalue"].round(4).to_string())

### 3.3 Estimation — full sample

We estimate AR(1)–GJR-GARCH(1,1) with Student-t innovations for each of the six assets on the full sample (1990–2026). The diagnostic protocol runs immediately after each estimation. Results feed into the DCC model in Part 4 — any diagnostic failure must be addressed before proceeding.

In [ ]:
# AR order per asset based on diagnostic results
# S&P500, US HY, Gold: LB(z) still failing with AR(1) → try AR(2)
# US IG: LB(z) failing with AR(1) → try AR(2)
# Eurostoxx 50, Oil: AR(1) sufficient

AR_SPECS = {
    "S&P500"      : {"ar1": True,  "ar2": True},
    "Eurostoxx 50": {"ar1": True,  "ar2": False},
    "US IG Bonds" : {"ar1": True,  "ar2": True},
    "US HY Bonds" : {"ar1": True,  "ar2": True},
    "Gold"        : {"ar1": True,  "ar2": True},
    "Oil futures" : {"ar1": True,  "ar2": False},
}

print("AR order per asset:")
for col, spec in AR_SPECS.items():
    order = "AR(2)" if spec["ar2"] else "AR(1)"
    print(f"  {col:<20} {order}")

In [ ]:
fits_full = {}
diag_full = []

for col in ASSETS:
    fit = fit_gjr_garch(returns[col].dropna(),
                        label=f"{col} | full",
                        ar1=AR_SPECS[col]["ar1"],
                        ar2=AR_SPECS[col]["ar2"])
    fits_full[col] = fit
    diag = run_garch_diagnostics(fit, asset=col, period="full")
    diag_full.append(diag)

diag_full = pd.concat(diag_full, ignore_index=True)